# Inverse Noise/SNR Sweep

Runs `wCSET`, `CSET`, `MCE`, `MNE`, `sLORETA` and `dSPM` on every run of the noise/SNR
sweep dataset (`sub-simf10-noisesweep`, generated via
`mindmap.analysis.simulation.generate_simulation_dataset` -- see
`tutorials/analysis/simulations/simulation_spike.ipynb`'s "Noise/SNR sweep dataset"
section) and scores each method's accuracy as a function of `target_snr`. Companion to
`inverse-simf.ipynb` (single run, all solvers) and `harmonic-prior-tuning.ipynb` (one
solver's hyperparameter swept across runs) -- this notebook instead sweeps `target_snr`
across runs, with `method` as the second axis.

The dataset also contains a noise-only condition (`target_snr=NaN`, zero-amplitude
signal), which this notebook does not evaluate: the low end of the SNR sweep already
covers "signal indistinguishable from noise" without the added complexity of a
noise-only condition (whose noise-free `gold` is exactly zero everywhere, so there's no
well-defined rising phase/peak to score against).

`generate_simulation_dataset` only writes `simulation_index.tsv` once *after* every run
in the sweep has completed (see `mindmap/analysis/simulation/pipeline.py`), so a
missing/stale index there means the dataset generation isn't finished yet. This notebook
is written against that file's expected schema and left **unexecuted** for that reason --
run it once the index exists and looks complete.

## 🧪 - Experiment setup
- `fwd`/`eig`: subject-level anatomy artifacts (forward solution, connectome Laplacian
  eigendecomposition), loaded once from `sub="simf10"`. This is the *inversion* forward
  (`desc="inv"`), deliberately different from the *simulation* forward the dataset was
  generated with (inverse-crime avoidance) -- only used for solving, never for projecting
  `gold`
- `simulator_dir`: path to the saved simulator (not loaded here -- `run_parallel_pipeline`
  loads it fresh inside each worker, since only workers need the live object); `hm.fwd`
  is the *simulation*-time forward, whose source space matches `gold`, used only to
  project `gold` back to sensor space for timing (see Data Processing below)
- `df_sweep`: `simulation_index.tsv` for `sub-simf10-noisesweep`, filtered to the
  continuous-SNR rows (`target_snr` not NaN) -- one row per run, used to build the
  `InverseSimTask` list below (`ave`/`cov`/`gold`/`source` paths + `target_snr` per run)

In [1]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mindmap as mm
import mne

from mindmap.analysis.functional.common.metric import find_peak_localization_error
from mindmap.analysis.functional.inverse import InverseSimTask, run_parallel_pipeline

# readers
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"
sub_sweep = "simf10-noisesweep"

# subject-level data (run-independent)
fwd = dm.load(sub, "fwd", desc="inv")
eig = dm.load(sub, "eig")
dipole_positions = mm.anatomy.source_space.get_dipole_positions(fwd["src"]) * 1000

# path to the saved simulator -- loaded fresh inside each worker by
# run_parallel_pipeline, not here, since only workers need the live object
simulator_dir = dm.roots["sim"]/f"sub-{sub_sweep}"/"simulator"

# simulation index -- one row per run. `target_snr` is NaN for the noise-only condition
# (snr_targets=None in generate_simulation_dataset); this notebook only evaluates the
# continuous-SNR sweep (see intro cell above for why noise-only is skipped).
df_index = pd.read_csv(dm.roots["sim"]/f"sub-{sub_sweep}"/"simulation_index.tsv", sep="\t")
df_sweep = df_index[df_index["target_snr"].notna()].copy()

print(f"{len(df_sweep)} runs across the SNR sweep (of {len(df_index)} total in the dataset).")

240 runs across the SNR sweep (of 245 total in the dataset).


## 🧬 - Data Processing

Runs the sweep through `mm.analysis.inverse.run_parallel_pipeline` (parallelized via
`joblib`, `n_jobs=4`) instead of a sequential per-run loop, composed from three
independent, freely-swappable stages:

- **`preprocess_hrp_pp`**: picks `hrp`/`pp` (half-rising-phase and peak-phase) from that
  run's own noise-free `gold`, projected to sensor space with the *simulation* forward
  (`ctx["simulator"].hm.fwd`, loaded fresh per run inside the worker -- matches `gold`'s
  source space, unlike the *inversion* `fwd` used for solving) -- immune to noise/SNR by
  construction, unlike finding the peak in the noisy `ave` directly. Builds the
  two-timepoint `phase_data` (column 0 = `hrp`, column 1 = `pp`) that actually gets
  solved, instead of the whole window, since only these two phases are ever scored.
- **`make_reconstruct(fwd, eig)`**: solves all six methods on `phase_data` (the actual
  noisy `ave`, not the clean projection). `MNE`/`sLORETA`/`dSPM` share one inverse
  operator and the same auto-estimated `lambda2` as `wCSET`; `CSET`/`MCE` use a fixed
  `lambda2=0.1` -- same split as `inverse-simf.ipynb`.
- **`postprocess_score`**: scores each solution against that run's own `gold` (indexed at
  the same `hrp`/`pp` sample indices) with five scalars -- `correlation_hrp`/`_pp`,
  `localization_error_hrp`, `ap_hrp`/`_pp` -- returned per run and registered into one
  `mm.metric.MetricSuite`, tagged with `method` and `snr` so results can be
  grouped/plotted along both axes.

`run_parallel_pipeline` hardcodes loading `ave`/`ncov`/`gold`/`src`/`simulator` from each
`InverseSimTask`'s paths inside the worker (never via a live `DatasetManager`, which isn't
picklable across processes) -- the three stages above only ever touch already-loaded,
in-memory objects.

In [2]:
def preprocess_hrp_pp(ctx):
    ave, gold, simulator = ctx["ave"], ctx["gold"], ctx["simulator"]

    # hrp/pp picked from this run's own noise-free gold, projected to sensor space with
    # the *simulation* forward (simulator.hm.fwd matches gold's source space; the `fwd`
    # used for solving is a deliberately different, inversion-time forward) -- immune to
    # noise/SNR by construction, unlike finding the peak in `ave` directly. No tmin/tmax
    # window needed: propagation is strictly causal and the spike waveform is zero
    # outside its own window, so the clean projection is zero everywhere except the
    # true peak.
    clean_ave = mne.apply_forward(simulator.hm.fwd, gold, ave.info)
    hrp_idx = mm.analysis.eeg.get_rising_phase(clean_ave, ratio=0.5, as_idx=True)[0]
    pp_idx = mm.analysis.eeg.get_rising_phase(clean_ave, ratio=1, as_idx=True)[0]

    # solve only at those two phases -- on the actual noisy `ave`, not the clean
    # projection -- instead of the whole cropped window
    phase_data = mne.EvokedArray(ave.data[:, [hrp_idx, pp_idx]], ave.info, tmin=0, nave=ave.nave, kind=ave.kind)

    return {"phase_data": phase_data, "hrp_idx": hrp_idx, "pp_idx": pp_idx}


def make_reconstruct(fwd, eig):
    """Solves all six inverse methods on the two-timepoint phase_data."""
    mne_methods = {"sloreta": "sLORETA", "dspm": "dSPM", "mne": "MNE"}

    def reconstruct(ctx):
        data, ncov = ctx["phase_data"], ctx["ncov"]

        # SNR estimated from the real, full-time-reference `ave` -- not `phase_data`,
        # whose tmin=0 fake time axis puts the real baseline window (negative time)
        # entirely outside its range. That silently produced an empty baseline slice
        # (NaN mean, "Mean of empty slice" warning), and crucially
        # max(0.01, 1.0/nan**2) evaluates to 0.01 (not NaN) in this argument order --
        # so lambda2_auto was silently pinned to the floor value every time, not
        # visibly broken. `ave` has the correct time reference, so this is fixed.
        snr, _ = mm.analysis.eeg.estimate_evoked_snr(
            evoked=ctx["ave"], baseline=ctx["simulator"].epoch_config.baseline, noise_cov=ncov)
        lambda2_auto = max(0.01, 1.0 / (snr.max() ** 2))

        inv_wcset = mm.analysis.inverse.wCSET(
            forward=fwd, noise_cov=ncov, depth=0.5,
            eigenvectors=eig["vec"], eigenvalues=eig["val"],
            harm=5, harmonic_method="lowpass",
        )
        inv_cset = mm.analysis.inverse.CSET(forward=fwd, noise_cov=ncov, depth=0.8, eigenvectors=eig["vec"])
        inv_mce = mm.analysis.inverse.MCE(forward=fwd, noise_cov=ncov, depth=0.8)
        inv_mne_op = mne.minimum_norm.make_inverse_operator(
            info=data.info, forward=fwd, noise_cov=ncov, fixed=True, depth=None, verbose="warning",
        )

        solutions = {
            "wcset": inv_wcset.solve(data, lambda2_auto),
            "cset": inv_cset.solve(data, lambda2=0.1),
            "mce": inv_mce.solve(data, lambda2=0.1),
        }
        for m, mne_name in mne_methods.items():
            solutions[m] = mne.minimum_norm.apply_inverse(data, inv_mne_op, lambda2_auto, method=mne_name, verbose="warning")

        return {"solutions": solutions}

    return reconstruct


def postprocess_score(ctx):
    """Scores each of the six solutions against this run's own gold, returns 5 scalars/method."""
    gold, hrp_idx, pp_idx = ctx["gold"], ctx["hrp_idx"], ctx["pp_idx"]
    src, simulator, task = ctx["src"], ctx["simulator"], ctx["task"]

    true_source_mask = src.select_targets(simulator.hm.src, simulator.hm.parc, simulator.hm.ras2tkras)
    gold_hrp, gold_pp = gold.data[:, hrp_idx], gold.data[:, pp_idx]

    rows = []
    for method, sol in ctx["solutions"].items():
        sol_hrp, sol_pp = sol.data[:, 0], sol.data[:, 1]

        # peak localization error at half-rising-phase only, same scope as harmonic-prior-tuning.ipynb
        hit = find_peak_localization_error(sol_hrp, true_source_mask, dipole_positions)

        rows.append({
            "method": method,
            "snr": task.target_snr,
            "correlation_hrp": np.corrcoef(gold_hrp, sol_hrp)[0, 1],
            "correlation_pp": np.corrcoef(gold_pp, sol_pp)[0, 1],
            "localization_error_hrp": hit["distance_mm"],
            "ap_hrp": mm.core.calc_pr_curve(gold_hrp > 0, sol_hrp)["ap"],
            "ap_pp": mm.core.calc_pr_curve(gold_pp > 0, sol_pp)["ap"],
        })
    return rows

In [3]:
tasks = [
    InverseSimTask(
        run=int(row["run"]),
        simulator_dir=simulator_dir,
        fave=dm.get_path(sub_sweep, "ave", run=int(row["run"])),
        fcov=dm.get_path(sub_sweep, "cov", run=int(row["run"])),
        fgold=dm.get_path(sub_sweep, "gold", run=int(row["run"])),
        fsource=dm.get_path(sub_sweep, "source", run=int(row["run"])),
        target_snr=row["target_snr"],
    )
    for _, row in df_sweep.iterrows()
]

results = run_parallel_pipeline(
    tasks, preprocess_hrp_pp, make_reconstruct(fwd, eig), postprocess_score, n_jobs=4,
)

metrics = mm.metric.MetricSuite(f"Noise/SNR sweep {sub_sweep}")
for run_rows in results:
    for row in run_rows:
        method, snr = row["method"], row["snr"]
        metrics.register(method, "correlation_hrp", row["correlation_hrp"], method=method, snr=snr)
        metrics.register(method, "correlation_pp", row["correlation_pp"], method=method, snr=snr)
        metrics.register(method, "localization_error_hrp", row["localization_error_hrp"], method=method, snr=snr)
        metrics.register(method, "ap_hrp", row["ap_hrp"], method=method, snr=snr)
        metrics.register(method, "ap_pp", row["ap_pp"], method=method, snr=snr)

pipeline:   0%|          | 0/240 [00:00<?, ?it/s]

pipeline:   3%|▎         | 8/240 [00:07<03:25,  1.13it/s]

pipeline:   5%|▌         | 12/240 [00:20<07:16,  1.91s/it]

pipeline:   7%|▋         | 16/240 [01:05<20:05,  5.38s/it]

pipeline:   8%|▊         | 20/240 [01:36<22:43,  6.20s/it]

pipeline:  10%|█         | 24/240 [01:59<21:53,  6.08s/it]

... (51 outputs left out)

pipeline:  97%|█████████▋| 232/240 [27:11<00:11,  1.38s/it]

pipeline:  98%|█████████▊| 236/240 [27:17<00:05,  1.42s/it]

pipeline: 100%|██████████| 240/240 [27:23<00:00,  1.45s/it]

pipeline: 100%|██████████| 240/240 [27:23<00:00,  6.85s/it]

## 📊 - Visualization

Correlation, average precision, and localization error vs. `target_snr` -- one line per
method for each of the five scalar metrics (`correlation_hrp`/`correlation_pp`,
`ap_hrp`/`ap_pp`, `localization_error_hrp`), median +/- 50% percentile interval across the
5 repetitions x N regions at each SNR level, same aggregation convention as
`harmonic-prior-tuning.ipynb`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
metrics.plot(metric="correlation_hrp", x="snr", kind="line", estimator="median", errorbar=("pi", 50), ax=axes[0])
metrics.plot(metric="correlation_pp", x="snr", kind="line", estimator="median", errorbar=("pi", 50), ax=axes[1])
fig.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
metrics.plot(metric="ap_hrp", x="snr", kind="line", estimator="median", errorbar=("pi", 50), ax=axes[0])
metrics.plot(metric="ap_pp", x="snr", kind="line", estimator="median", errorbar=("pi", 50), ax=axes[1])
fig.tight_layout()


In [ ]:
metrics.plot(
    metric = "localization_error_hrp",
    x = "snr",
    kind = "line",
    estimator = "median",
    errorbar = ("pi", 50),
);
